# Evaluation — relaxed exact match per model and prompting strategy

Same slots, same notes, same TP/FP/FN rule as `5_evaluate_metrics.ipynb`. The only
difference is the canonicalisation applied to both sides before the equality test.

* **Strict** — surface normalisation only: whitespace collapsed, `null` treated as absent.
* **Relaxed** — the above plus the per-key rules in `RELAXED_CANONICALISERS`.

Currently one rule is declared: `frequency` is not penalised for repeating an indication
that the same medication already extracted correctly into `indication`, so
`"Q12H:PRN anxiety"` matches an annotation of `"Q12H:PRN"` when `indication` is
`["anxiety"]`. The rule fires only on that duplication — a frequency such as `"Q12H PRN"`
is left untouched, and if `indication` is empty nothing is forgiven.

The criterion is a *choice of measurement*, not a repair of the model output: the JSON on
disk is never rewritten. Strict numbers are recomputed here so both criteria can be diffed
in one session; they must reproduce `5_evaluate_metrics.ipynb` exactly.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

from clinical_notes_extraction.utils.llm.evaluation import (  # noqa: E402
    ATTRIBUTES,
    TOP_LEVEL_KEYS,
    RELAXED_CANONICALISERS,
    evaluate_all,
    load_ground_truth_dir,
    load_results_tree,
    normalise,
)

from clinical_notes_extraction.utils.llm.evaluation import (  # noqa: E402
    metrics_table, metrics_to_docx, metrics_to_html, metrics_to_latex,
)

In [ ]:
NOTEBOOK_DIR = Path.cwd()
ROOT = next(p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("project root:", ROOT)

## 1. Paths

Same `RUN_ID` as the exact-match notebook: the two criteria must be read off the same
extraction run, otherwise the comparison is meaningless.

In [ ]:
DATA_DIR = NOTEBOOK_DIR / "data"

SPLIT = "dev"          # ground_truth/dev | ground_truth/prod
CRITERION = "relaxed_exact_match"        # "exact_match" no notebook 5

GROUND_TRUTH_DIR = DATA_DIR / "annotations" / "ground_truth" / SPLIT
RESULTS_ROOT = DATA_DIR / "llm_extraction_results" / SPLIT
RUN_ID = "20260731_014511"
RUN_DIR = RESULTS_ROOT / RUN_ID

OUTPUT_DIR = RESULTS_ROOT / "evaluation" / RUN_ID / CRITERION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CASEFOLD = False       # keep in sync with 5_evaluate_metrics.ipynb

assert GROUND_TRUTH_DIR.is_dir(), GROUND_TRUTH_DIR
assert RUN_DIR.is_dir(), RUN_DIR

## 2. Load

Coverage is a property of the run, not of the criterion, so it is not repeated here —
check section 3 of the exact-match notebook before reading any number below.

In [ ]:
gold_by_id = load_ground_truth_dir(GROUND_TRUTH_DIR)
records = load_results_tree(RUN_DIR)

print(f"{len(gold_by_id)} annotated notes in {SPLIT}")
print(f"{len(records)} result files in run {RUN_ID}")

## 3. Score under both criteria

`canonicalisers=None` is the strict criterion and reproduces the exact-match numbers
bit for bit.

In [ ]:
strict = evaluate_all(gold_by_id, records, casefold=CASEFOLD)
results = evaluate_all(gold_by_id, records, casefold=CASEFOLD,
                       canonicalisers=RELAXED_CANONICALISERS)

print("declared relaxed keys:", sorted(RELAXED_CANONICALISERS))
print("cells:", len(results))

## 4. Verification — the relaxation must touch one key only

Every key without a declared rule has to be identical to the integer under both criteria.
If this fails, the rule is firing where it should not and nothing below can be reported.

In [ ]:
ALL_KEYS = list(TOP_LEVEL_KEYS) + list(ATTRIBUTES)
RELAXED_KEYS = set(RELAXED_CANONICALISERS)

problems = []
for cell in strict:
    for key in ALL_KEYS:
        if key in RELAXED_KEYS:
            continue
        s, r = strict[cell].per_key.get(key), results[cell].per_key.get(key)
        s = (s.tp, s.fp, s.fn) if s else (0, 0, 0)
        r = (r.tp, r.fp, r.fn) if r else (0, 0, 0)
        if s != r:
            problems.append((cell, key, s, r))

assert not problems, problems
print("ok — only", ", ".join(sorted(RELAXED_KEYS)), "differs between criteria")

### How much moved

Slots that changed classification. Each recovered slot was one FP *and* one FN under the
strict criterion, so it improves precision and recall together. A large count here would
mean the rule is carrying more of the result than a methods footnote can justify.

In [ ]:
rows = []
for (model, strategy), run in sorted(results.items()):
    for key in sorted(RELAXED_KEYS):
        s, r = strict[(model, strategy)].per_key.get(key), run.per_key.get(key)
        if s is None or r is None:
            continue
        rows.append({
            "model": model, "strategy": strategy, "key": key,
            "slots": s.tp + s.fp + s.fn,
            "recovered": r.tp - s.tp,
            "f1_strict": s.f1, "f1_relaxed": r.f1, "delta_f1": r.f1 - s.f1,
        })

moved = pd.DataFrame(rows).set_index(["model", "strategy", "key"])
moved.round(3)

## 5. Headline table — errors counted as no output

In [ ]:
runs = (
    pd.DataFrame([run.record() for run in results.values()])
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)
runs[["model", "strategy", "precision", "recall", "f1",
      "tp", "fp", "fn", "support", "n_notes", "invalid_rate"]]

In [ ]:
for metric in ["precision", "recall", "f1"]:
    print(f"\n=== {metric} ===")
    print(runs.pivot(index="model", columns="strategy", values=metric).round(3))

## 6. Strict vs relaxed

Same shape as section 5 of the exact-match notebook, where `f1_casefold` and `gap_casing`
already isolate one relaxation. `gap_relaxed` isolates the indication-duplication rule.

What matters is the **ranking**, not the absolute gain. If the order of the cells is the
same under both criteria, the conclusion does not depend on the choice — say so in the
discussion. If it changes, the reordering is itself a finding about which models produce
verbose-but-correct output rather than wrong output.

In [ ]:
def f1_series(res, name):
    return (
        pd.DataFrame([r.record() for r in res.values()])
        .set_index(["model", "strategy"])["f1"].rename(name)
    )

comparison = (
    runs.set_index(["model", "strategy"])[["f1", "invalid_rate"]]
    .rename(columns={"f1": "f1_relaxed"})
    .join(f1_series(strict, "f1_strict"))
)
comparison["gap_relaxed"] = comparison.f1_relaxed - comparison.f1_strict
comparison["rank_strict"] = comparison.f1_strict.rank(ascending=False).astype(int)
comparison["rank_relaxed"] = comparison.f1_relaxed.rank(ascending=False).astype(int)

comparison = comparison[["f1_strict", "f1_relaxed", "gap_relaxed",
                         "rank_strict", "rank_relaxed", "invalid_rate"]]
comparison.sort_values("f1_relaxed", ascending=False).round(3)

In [ ]:
# does the relaxation reorder the grid at all?
reordered = comparison[comparison.rank_strict != comparison.rank_relaxed]
print("cells whose rank changes:", len(reordered))
reordered.sort_values("rank_relaxed")

## 7. Breakdown per key

In [ ]:
per_key = pd.DataFrame([row for run in results.values() for row in run.per_key_records()])

per_key.pivot_table(index="key", columns=["model", "strategy"], values="f1").reindex(
    list(TOP_LEVEL_KEYS) + list(ATTRIBUTES)
).round(3)

In [ ]:
# the relaxed key, strict vs relaxed, side by side across the whole grid
strict_key = pd.DataFrame(
    [row for run in strict.values() for row in run.per_key_records()]
)

for key in sorted(RELAXED_KEYS):
    print(f"\n=== {key} ===")
    print(
        strict_key[strict_key.key == key]
        .set_index(["model", "strategy"])[["f1"]].rename(columns={"f1": "strict"})
        .join(per_key[per_key.key == key]
              .set_index(["model", "strategy"])[["f1"]].rename(columns={"f1": "relaxed"}))
        .round(3).to_string()
    )

## 8. Breakdown per note

In [ ]:
per_note = pd.DataFrame([row for run in results.values() for row in run.per_note_records()])

per_note.groupby("note_id").agg(
    mean_f1=("f1", "mean"),
    failed_runs=("valid", lambda s: (~s).sum()),
    n_gold_medications=("n_gold_medications", "max"),
).sort_values("mean_f1").head(10).round(3)

### Which slots disagree, for one run and one note

Values are shown as the relaxed criterion sees them, so a row that disappears relative to
the exact-match notebook is a slot the relaxation forgave.

In [ ]:
def canonicalise(key, value, attrs):
    """The relaxed criterion's view of one value: per-key rule, then normalise."""
    rule = RELAXED_CANONICALISERS.get(key)
    return normalise(rule(value, attrs) if rule else value, casefold=CASEFOLD)


MODEL, STRATEGY = runs.loc[0, "model"], runs.loc[0, "strategy"]
NOTE_ID = per_note[(per_note.model == MODEL) & (per_note.strategy == STRATEGY)] \
    .sort_values("f1").iloc[0].note_id

prediction = next(
    (r.get("output") or {} for r in records
     if r["model"] == MODEL and r["strategy"] == STRATEGY and r["note_id"] == NOTE_ID),
    {},
)

gold_note = gold_by_id[NOTE_ID]
rows = [
    {"medication": "-", "key": key,
     "gold": canonicalise(key, gold_note.get(key), gold_note),
     "predicted": canonicalise(key, prediction.get(key), prediction)}
    for key in TOP_LEVEL_KEYS
]

gold_meds = gold_note.get("medications") or []
pred_meds = prediction.get("medications") or []
for i in range(max(len(gold_meds), len(pred_meds))):
    g = (gold_meds[i] if i < len(gold_meds) else {}).get("attributes", {})
    p = (pred_meds[i] if i < len(pred_meds) else {}).get("attributes", {})
    for key in ATTRIBUTES:
        rows.append({"medication": i + 1, "key": key,
                     "gold": canonicalise(key, g.get(key), g),
                     "predicted": canonicalise(key, p.get(key), p)})

diff = pd.DataFrame(rows)
print(f"{MODEL} / {STRATEGY} — note {NOTE_ID} (relaxed criterion)")
diff[diff.gold != diff.predicted]

## 9. Final tables — one row per key, P/R/F1 per model and prompting strategy

Identical layout to the exact-match document so the two can be read side by side in the
thesis. Only the caption and the criterion change.

In [ ]:
table_all = metrics_table(results)
table_all.map("{:.2f}".format)

In [ ]:
MODELS = sorted({model for model, _ in results})
STRATEGIES = sorted({strategy for _, strategy in results})

for model in MODELS:
    print(f"\n=== {model} ===")
    print(metrics_table(results, models=[model]).map("{:.2f}".format).to_string())

In [ ]:
for strategy in STRATEGIES:
    print(f"\n=== {strategy} ===")
    print(metrics_table(results, strategies=[strategy]).map("{:.2f}".format).to_string())

In [ ]:
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(exist_ok=True)

def slug(name: str) -> str:
    return name.replace(":", "_").replace(".", "_")

headline = runs.set_index(["model", "strategy"])[
    ["precision", "recall", "f1", "invalid_rate"]
]
headline.columns = ["P", "R", "F1", "invalid"]

# (filename, caption, dataframe) — file names prefixed so they do not overwrite
# the exact-match HTML exports written into the same directory
exports = [
    ("relaxed_summary",
     "Table 1: Relaxed exact-match Precision (P), Recall (R) and F1 per model and "
     "prompting strategy, over all annotated notes. Failed extractions counted as "
     "missing output.", headline),
    ("relaxed_criterion_comparison",
     "Table 2: Strict versus relaxed exact-match F1 per model and prompting strategy.",
     comparison.sort_values("f1_relaxed", ascending=False).round(3)),
]
exports += [
    (f"relaxed_by_key_{slug(strategy)}",
     f"Table: relaxed exact-match results per key under {strategy} prompting.",
     metrics_table(results, strategies=[strategy]))
    for strategy in STRATEGIES
]
exports += [
    (f"relaxed_by_key_{slug(model)}",
     f"Table: relaxed exact-match results per key for {model}.",
     metrics_table(results, models=[model]))
    for model in MODELS
]
exports.append(("relaxed_appendix_full_grid",
                "Appendix table: every model and prompting strategy, relaxed criterion.",
                table_all))

tables = {caption: frame for _, caption, frame in exports}

DOCX_PATH = f"{TABLES_DIR}/relaxed_exact_match_tables_{RUN_ID}.docx"

try:
    metrics_to_docx(tables, DOCX_PATH,
                    title=f"Relaxed exact-match evaluation — run {RUN_ID}")
    print("wrote", DOCX_PATH)
except ImportError:
    print("python-docx not installed (pip install python-docx) — writing HTML instead")

for name, caption, frame in exports:
    (TABLES_DIR / f"{name}.html").write_text(
        metrics_to_html(frame, caption=caption), encoding="utf-8"
    )

print(sorted(p.name for p in TABLES_DIR.iterdir()))

## 10. Export the raw numbers

In [ ]:
runs.to_csv(OUTPUT_DIR / "relaxed_metrics_per_run.csv", index=False)
per_key.to_csv(OUTPUT_DIR / "relaxed_metrics_per_key.csv", index=False)
per_note.to_csv(OUTPUT_DIR / "relaxed_metrics_per_note.csv", index=False)
comparison.to_csv(OUTPUT_DIR / "criterion_comparison.csv")
moved.to_csv(OUTPUT_DIR / "relaxed_slots_recovered.csv")

print("written to", OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.glob("*.csv")))

In [ ]:
import json
import collections
from pathlib import Path

GT_DIR = Path("data/annotations/ground_truth/development")

def is_filled(value):
    """Um slot conta como preenchido se não for null nem vazio."""
    if value is None:
        return False
    if isinstance(value, str) and not value.strip():
        return False
    if isinstance(value, (list, dict)) and not value:
        return False
    return True

med_filled = collections.Counter()
med_keys = set()
note_filled = collections.Counter()
note_keys = set()
n_notes = 0
n_meds = 0

for path in sorted(GT_DIR.glob("*.json")):
    ann = json.loads(path.read_text(encoding="utf-8"))
    n_notes += 1

    meds = ann.get("medications", [])
    for med in meds:
        n_meds += 1
        med_keys.update(med.keys())
        for key, value in med.items():
            if is_filled(value):
                med_filled[key] += 1

    for key, value in ann.items():
        if key == "medications":
            continue
        note_keys.add(key)
        if is_filled(value):
            note_filled[key] += 1

print(f"{n_notes} notas, {n_meds} medicamentos\n")

print(f"{'campo (por medicamento)':34s} {'preench.':>9} {'total':>7} {'%':>7}")
for key in sorted(med_keys):
    n = med_filled[key]
    print(f"{key:34s} {n:9d} {n_meds:7d} {n/n_meds:7.0%}")

if note_keys:
    print(f"\n{'campo (por nota)':34s} {'preench.':>9} {'total':>7} {'%':>7}")
    for key in sorted(note_keys):
        n = note_filled[key]
        print(f"{key:34s} {n:9d} {n_notes:7d} {n/n_notes:7.0%}")